# 03 — Tokenizers & ModelConfig

Covers:
- `myllm.tokenizers.get_tokenizer()` — unified factory
- `GPT2Tokenizer` — direct usage & token inspection
- `ModelConfig` — create, inspect, save/load, estimate memory

> **Colab users:** Run the setup cell, restart runtime, then continue.

In [ ]:
import os, sys, subprocess

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _install():
    IN_COLAB = _is_colab()
    if IN_COLAB:
        repo = 'MyLLM'
        if not os.path.exists(repo):
            print('Cloning repository...')
            r = subprocess.run(
                ['git', 'clone', 'https://github.com/silvaxxx1/MyLLM.git', repo],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                print('Clone failed:\n', r.stderr); return
        print('Installing myllm...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', f'./{repo}'],
            capture_output=True, text=True
        )
    else:
        root = os.path.abspath(os.path.join(os.getcwd(), '..'))
        print(f'Installing from {root} ...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-e', root],
            capture_output=True, text=True
        )
    if r.returncode != 0:
        print('Install FAILED. Error output:')
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
    else:
        msg = 'Restart runtime now (Runtime → Restart runtime).' if IN_COLAB else 'Ready.'
        print('Done!', msg)

try:
    import myllm
    print(f'myllm {myllm.__version__} already installed.')
except ImportError:
    _install()

## 1. Tokenizer factory

In [ ]:
from myllm.tokenizers import get_tokenizer, list_available_models

print('Registered tokenizer families:')
for name in list_available_models():
    print(' ', name)

In [ ]:
tok  = get_tokenizer('gpt2')
text = 'Hello, world! The future of AI is bright.'
ids  = tok.encode(text)
back = tok.decode(ids)

print('Text      :', text)
print('Token IDs :', ids)
print('Decoded   :', back)
print('# tokens  :', len(ids))

## 2. GPT-2 tokenizer — direct usage

In [ ]:
from myllm.tokenizers import GPT2Tokenizer

tok = GPT2Tokenizer()

sentences = [
    'The transformer architecture was introduced in 2017.',
    'Attention is all you need.',
    'Language models predict the next token.',
]

print(f'{"Sentence":<50}  Tokens')
print('-' * 60)
for s in sentences:
    print(f'{s:<50}  {len(tok.encode(s))}')

In [ ]:
# Inspect individual token pieces
text = 'GPT-2 uses byte-pair encoding tokenization.'
ids  = tok.encode(text)

print('Token breakdown:')
for token_id in ids:
    piece = tok.decode([token_id])
    print(f'  {token_id:>6}  {repr(piece)}')

## 3. ModelConfig — create and inspect

In [ ]:
from myllm import ModelConfig

cfg = ModelConfig.from_name('gpt2-medium')

print('Architecture')
for attr in ['name', 'n_layer', 'n_head', 'n_embd', 'head_size',
             'block_size', 'vocab_size']:
    print(f'  {attr:<20} = {getattr(cfg, attr)}')

print('\nComponents')
for attr in ['norm_class_name', 'mlp_class_name', 'activation',
             'position_embedding', 'use_rope', 'weight_tying']:
    print(f'  {attr:<20} = {getattr(cfg, attr)}')

print('\nHyperparameters')
for attr in ['learning_rate', 'weight_decay', 'dropout']:
    print(f'  {attr:<20} = {getattr(cfg, attr)}')

## 4. Memory estimation across all GPT-2 sizes

In [ ]:
import torch
from myllm import ModelConfig

models = ['gpt2-small', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']

print(f'{"Model":<16}  {"Params":>10}  {"FP32 (GB)":>10}  {"FP16 (GB)":>10}')
print('-' * 55)
for name in models:
    cfg  = ModelConfig.from_name(name)
    fp32 = cfg.estimate_memory(dtype=torch.float32)
    fp16 = cfg.estimate_memory(dtype=torch.float16)
    print(f'{name:<16}  {fp32["n_parameters"]/1e6:>9.1f}M  '
          f'{fp32["parameters_gb"]:>10.2f}  {fp16["parameters_gb"]:>10.2f}')

## 5. Save & load a config

In [ ]:
import json, tempfile, os
from myllm import ModelConfig

cfg = ModelConfig.from_name('gpt2-small')

with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as f:
    path = f.name

cfg.save(path)
print('Saved to:', path)

with open(path) as f:
    data = json.load(f)
print('JSON keys:', list(data.keys())[:8], '...')

loaded = ModelConfig.load(path)
assert loaded.n_layer == cfg.n_layer
print('\nLoaded back. n_layer =', loaded.n_layer)

os.unlink(path)

## 6. Custom config via `update()`

In [ ]:
import torch
from myllm import ModelConfig

cfg = ModelConfig.from_name('gpt2-small')
cfg.update(n_layer=6, n_head=8, n_embd=512, dropout=0.0, learning_rate=1e-3)

mem = cfg.estimate_memory(dtype=torch.float32)
print(f'Tiny custom model: n_layer={cfg.n_layer}, n_head={cfg.n_head}, n_embd={cfg.n_embd}')
print(f'{mem["n_parameters"]/1e6:.1f}M params   {mem["parameters_gb"]:.3f} GB')

## 7. Trainable parameters

In [ ]:
from myllm import ModelConfig

cfg = ModelConfig.from_name('gpt2-small')
print('Trainable config params (int / float / bool):')
for k, v in cfg.get_trainable_params().items():
    print(f'  {k:<30} = {v}')